In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# CONFIG
# =========================
DATA_PATH = "../transactional/DataCoSupplyChainDataset.csv"
DESC_PATH = "../transactional/DescriptionDataCoSupplyChain.csv"  # optional
OUT_DIR = Path("../fraud_preprocessing")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_NAME = "fraud"
LOW_CARDINALITY_THRESHOLD = 15


In [ ]:

# =========================
# LOAD
# =========================
df = pd.read_csv(DATA_PATH, encoding="latin-1", low_memory=False)

# Optional: load description file if needed for reference
try:
    desc_df = pd.read_csv(DESC_PATH, encoding="latin-1", low_memory=False)
except Exception:
    desc_df = None

print("Original shape:", df.shape)
print("Columns:", len(df.columns))


In [ ]:

# =========================
# 1) TARGET CREATION
# fraud = 1 when Order Status == SUSPECTED_FRAUD
# =========================
if "Order Status" not in df.columns:
    raise ValueError("Expected column 'Order Status' was not found.")

df[TARGET_NAME] = (df["Order Status"].astype(str).str.upper() == "SUSPECTED_FRAUD").astype(int)


In [ ]:

# =========================
# 2) COLUMN AUDIT
# =========================
audit_rows = []

for col in df.columns:
    audit_rows.append({
        "column": col,
        "dtype": str(df[col].dtype),
        "n_unique": int(df[col].nunique(dropna=True)),
        "missing_count": int(df[col].isna().sum()),
        "missing_pct": float(df[col].isna().mean() * 100)
    })

audit_df = pd.DataFrame(audit_rows)


In [ ]:

# =========================
# 3) DROP OBVIOUS NON-MODELING / LEAKAGE / REDUNDANT COLUMNS
# =========================
# Notes:
# - leakage/post-outcome fields are removed
# - IDs / PII / operational keys are removed
# - only columns that exist are dropped

drop_candidates = [
    # direct leakage / post-event outcome
    "Order Status",
    "Delivery Status",
    "Late_delivery_risk",
    "Days for shipping (real)",
    "shipping date (DateOrders)",

    # likely identifiers / keys
    "Type",
    "Order Id",
    "Order Item Id",
    "Product Card Id",
    "Customer Id",
    "Department Id",
    "Category Id",
    "Order Customer Id",

    # PII / low generalization fields
    "Customer Email",
    "Customer Password",
    "Customer Fname",
    "Customer Lname",
    "Customer Street",
    "Order Zipcode",

    # likely duplicate / near-duplicate geography text
    "Product Description",
]

drop_existing = [c for c in drop_candidates if c in df.columns]

# Drop all-missing columns
all_missing_cols = [c for c in df.columns if df[c].isna().all()]

# Drop constant columns
constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1 and c != TARGET_NAME]

# Drop exact duplicate columns
duplicate_cols = []
seen = {}
for col in df.columns:
    if col == TARGET_NAME:
        continue
    values_key = tuple(df[col].fillna("__MISSING__").astype(str).values[:5000])  # light precheck
    if values_key in seen:
        other = seen[values_key]
        if df[col].fillna("__MISSING__").astype(str).equals(df[other].fillna("__MISSING__").astype(str)):
            duplicate_cols.append(col)
    else:
        seen[values_key] = col

cols_to_drop = sorted(set(drop_existing + all_missing_cols + constant_cols + duplicate_cols))
df_model = df.drop(columns=cols_to_drop, errors="ignore").copy()


In [ ]:

# =========================
# 4) DATE FEATURE ENGINEERING
# =========================
order_date_col = "order date (DateOrders)"
if order_date_col in df_model.columns:
    dt = pd.to_datetime(df_model[order_date_col], errors="coerce")

    df_model["order_year"] = dt.dt.year
    df_model["order_quarter"] = dt.dt.quarter
    df_model["order_month"] = dt.dt.month
    df_model["order_day"] = dt.dt.day
    df_model["order_weekday"] = dt.dt.weekday
    df_model["order_hour"] = dt.dt.hour
    df_model["order_is_weekend"] = dt.dt.weekday.isin([5, 6]).astype("Int64")

    df_model = df_model.drop(columns=[order_date_col])


In [ ]:

# =========================
# 5) MISSING VALUE HANDLING
# =========================
# Special handling for Customer Zipcode
if "Customer Zipcode" in df_model.columns:
    df_model["Customer Zipcode_was_missing"] = df_model["Customer Zipcode"].isna().astype(int)
    if pd.api.types.is_numeric_dtype(df_model["Customer Zipcode"]):
        df_model["Customer Zipcode"] = df_model["Customer Zipcode"].fillna(df_model["Customer Zipcode"].median())
    else:
        df_model["Customer Zipcode"] = df_model["Customer Zipcode"].fillna("Unknown")

# General numeric/object fill
numeric_cols = [c for c in df_model.select_dtypes(include=[np.number]).columns if c != TARGET_NAME]
categorical_cols = [c for c in df_model.columns if c not in numeric_cols + [TARGET_NAME]]

for col in numeric_cols:
    if df_model[col].isna().any():
        df_model[col] = df_model[col].fillna(df_model[col].median())

for col in categorical_cols:
    if df_model[col].isna().any():
        df_model[col] = df_model[col].fillna("Unknown")


In [ ]:

# =========================
# 6) BUILD NUMERIC-READY TABLE
# - one-hot encode low-cardinality categoricals
# - frequency encode high-cardinality categoricals
# =========================
categorical_cols = [c for c in df_model.select_dtypes(include=["object"]).columns if c != TARGET_NAME]
low_card_cols = [c for c in categorical_cols if df_model[c].nunique(dropna=False) <= LOW_CARDINALITY_THRESHOLD]
high_card_cols = [c for c in categorical_cols if c not in low_card_cols]

df_numeric = df_model.copy()

# Frequency encode high-cardinality categorical columns
freq_manifest = []
for col in high_card_cols:
    freq_map = df_numeric[col].value_counts(normalize=True, dropna=False)
    df_numeric[f"{col}__freq"] = df_numeric[col].map(freq_map).astype(float)
    freq_manifest.append({
        "column": col,
        "encoding": "frequency",
        "n_unique": int(df_model[col].nunique(dropna=False))
    })

df_numeric = df_numeric.drop(columns=high_card_cols)

# One-hot encode low-cardinality columns
ohe_manifest = []
if low_card_cols:
    for col in low_card_cols:
        ohe_manifest.append({
            "column": col,
            "encoding": "one_hot",
            "n_unique": int(df_model[col].nunique(dropna=False))
        })
    df_numeric = pd.get_dummies(df_numeric, columns=low_card_cols, drop_first=False, dtype=int)

# Final guard: ensure target is last column
feature_cols = [c for c in df_numeric.columns if c != TARGET_NAME]
df_numeric = df_numeric[feature_cols + [TARGET_NAME]]


In [ ]:

# =========================
# 7) MANIFESTS / SUMMARIES
# =========================
feature_manifest_rows = []

for col in df_model.columns:
    if col == TARGET_NAME:
        role = "target"
        encoding = "binary_target"
    elif col in numeric_cols or col in ["order_year", "order_quarter", "order_month", "order_day",
                                        "order_weekday", "order_hour", "order_is_weekend",
                                        "Customer Zipcode_was_missing"]:
        role = "feature"
        encoding = "numeric"
    elif col in low_card_cols:
        role = "feature"
        encoding = "one_hot"
    elif col in high_card_cols:
        role = "feature"
        encoding = "frequency"
    else:
        role = "feature"
        encoding = "unknown"

    feature_manifest_rows.append({
        "column": col,
        "role": role,
        "encoding": encoding,
        "dtype_before_numeric_table": str(df_model[col].dtype),
        "n_unique": int(df_model[col].nunique(dropna=False)),
        "missing_after_cleaning": int(df_model[col].isna().sum())
    })

feature_manifest_df = pd.DataFrame(feature_manifest_rows)

summary_df = pd.DataFrame([
    {"metric": "original_rows", "value": df.shape[0]},
    {"metric": "original_cols", "value": df.shape[1]},
    {"metric": "cleaned_base_rows", "value": df_model.shape[0]},
    {"metric": "cleaned_base_cols", "value": df_model.shape[1]},
    {"metric": "numeric_ready_rows", "value": df_numeric.shape[0]},
    {"metric": "numeric_ready_cols", "value": df_numeric.shape[1]},
    {"metric": "target_positive_count", "value": int(df_model[TARGET_NAME].sum())},
    {"metric": "target_positive_rate", "value": float(df_model[TARGET_NAME].mean())},
    {"metric": "dropped_columns_count", "value": len(cols_to_drop)},
    {"metric": "low_cardinality_categoricals", "value": len(low_card_cols)},
    {"metric": "high_cardinality_categoricals", "value": len(high_card_cols)},
])

drop_log_df = pd.DataFrame({
    "dropped_column": cols_to_drop
})

encoding_log_df = pd.concat([
    pd.DataFrame(ohe_manifest),
    pd.DataFrame(freq_manifest)
], ignore_index=True)


In [ ]:

# =========================
# 8) SAVE OUTPUTS
# =========================
base_path = OUT_DIR / "fraud_prepared_base.csv"
numeric_path = OUT_DIR / "fraud_prepared_numeric.csv"
audit_path = OUT_DIR / "fraud_original_column_audit.csv"
manifest_path = OUT_DIR / "fraud_feature_manifest.csv"
summary_path = OUT_DIR / "fraud_preprocessing_summary.csv"
drop_log_path = OUT_DIR / "fraud_dropped_columns.csv"
encoding_log_path = OUT_DIR / "fraud_encoding_log.csv"

df_model.to_csv(base_path, index=False)
df_numeric.to_csv(numeric_path, index=False)
audit_df.to_csv(audit_path, index=False)
feature_manifest_df.to_csv(manifest_path, index=False)
summary_df.to_csv(summary_path, index=False)
drop_log_df.to_csv(drop_log_path, index=False)
encoding_log_df.to_csv(encoding_log_path, index=False)

print("\nSaved files:")
print(base_path)
print(numeric_path)
print(audit_path)
print(manifest_path)
print(summary_path)
print(drop_log_path)
print(encoding_log_path)

print("\nFinal shapes:")
print("Cleaned base:", df_model.shape)
print("Numeric ready:", df_numeric.shape)

print("\nTarget distribution:")
print(df_model[TARGET_NAME].value_counts(dropna=False))
print("Fraud rate:", round(df_model[TARGET_NAME].mean(), 6))